In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/home/domans/dianne-codebase/')
import viewer
import dianne
import pandas as pd
import matplotlib.colors as mcolors
import scanpy as sc
dianne.setNotebookWidth(100)

idm = './identity-matrix.csv'
pd.DataFrame([[1,0,0], [0,1,0], [0,0,1]]).to_csv(idm, index=False, header=False)

In [ ]:
xdataPath1 = '/projects/chuang-lab/USERS/domans/test-atera/data-set1/'
xdataPath2 = '/projects/chuang-lab/USERS/domans/test-atera/data-set2/'
dataPath = '/projects/chuang-lab/USERS/domans/test-atera/output-STQ/'
samples = ['Breast_Cancer']

xdataPath = xdataPath1

F = 1
fname = f'img.data.ctranspath-{F}.h5ad'
xenium_to_he_matrices = {samples[0]: idm}
xenium_bundle_paths = {samples[0]: xdataPath}

imgs = {samples[0]: xdataPath + 'seg4.ome.tif'}

imtp = 'WTA_Preview_FFPE_Breast_Cancer_he_image.ome.tif'
secondary_images = {samples[0]: xdataPath1 + imtp}
secondary_matrices = {samples[0]: xdataPath + 'WTA_Preview_FFPE_Breast_Cancer_he_alignment.csv'}

dfa = pd.read_csv(xdataPath + 'WTA_Preview_FFPE_Breast_Cancer_cell_groups.csv', index_col=0)
index = sc.read_10x_h5(xdataPath + 'cell_feature_matrix.h5').obs.index
annotations = {samples[0]: pd.Categorical(dfa['group'].loc[index].values)}
annotationsPalette = dfa.set_index('group')['color'].to_dict()

classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)

patch_size = 8 # number of tiles, in each dimension, to include in a patch (e.g. 8 means 8x8=64 tiles per patch)
ts, mpp, tile_size = dianne.loadSTQParams(dataPath + samples[0], F)
ads, _, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne.loadDataAndPreparePatches(samples, dataPath, fname, L=None, ts=ts, mpp=mpp, N=patch_size)
sizes = {s: ads[s].shape[0] for s in samples}
print(f'Prepared {patchesCDFs.shape[0]} patches')

runfn = dianne.makeRunFn(patchCoordinates, ads, samples, qs, ts, mpp, tile_size=tile_size, patch_size=patch_size, PCMA_alpha=0.8, alpha_img=0.5, multiplier=2)
savefn = dianne.makeSaveFn(patchCoordinates, ads, samples, qs, ts, mpp, PCMA_alpha=0.8, tile_size=tile_size, patch_size=patch_size, body_overlap=0.25, classifierPaths=classifierPaths)
loadfn = dianne.makeLoadFn(classifierPaths)
listfn = dianne.makeListFn(classifierPaths)

drawings = viewer.create_viewer(samples, imgs, height="800px", run_inference_fn=runfn, sample_sizes=sizes,
                                xenium_mpp=0.2125, max_cells=10000, matrices=xenium_to_he_matrices, xenium_bundle_paths=xenium_bundle_paths,
                                secondary_images=secondary_images, secondary_matrices=secondary_matrices, draw_on_secondary=True,
                                annotations=annotations, category_colors=annotationsPalette,
                                save_func=savefn, load_func=loadfn, list_names_func=listfn)[1]